In [ ]:
import numpy as np
from datetime import datetime

def source_activity(file_date, source):
    '''
    Function that calculates the remaining activity of a radioisotope source 
    by applying its halflife and total time to its initial activity. 

    is dependant on the source type where the function chooses
    which hardcoded source values to use.
    __________________
    Parameters
        init_activity: Float
        The initial activity of the radio-isotope from when it was first calibrated.
        
    ___________________
    Outputs
        
    '''

    #here index 0 is Am, then Ba, then Co, then Cs
    init_activity = []
    source_age = [] # ages of each source
    HL3_confirmed = [] # half lifes of sources
    emission_frac = [] # emission fractions of sources

    # feed in parameters 
    
    time = source_age(file_date, source)
    idk_activity = init_activity*(0.5**(time/HL3_confirmed))

    activity = remaining_activity*emission_frac

    return activity



def eff_info_spe(filename):
    """
    Function that reads an .Spe file and returns date measured and the type of source.
    
    **NOTE** There is no information in the files that states what radio-isotope
    the file belongs to, other than the file naming convention used when
    collecting data. For this reason this function relys on that fact when returning
    the radio-isotope.
    """
    source = str(filename).split('_')[1]
    
    with open(filename, "r") as f:
        lines = [line.strip() for line in f]

    # $DATE_MEA: is followed by a line like "10/12/2026 16:18:58" 
    meas_date = lines[lines.index('$DATE_MEA:') + 1]

    return meas_date, source



def eff_info_mca(filename):
    """
    Function that reads an .mca file and returns date measured and the type of source.
    
    **NOTE** There is no information in the files that states what radio-isotope
    the file belongs to, other than the file naming convention used when
    collecting data. For this reason this function relys on that fact when returning
    the radio-isotope.
    """
    source = str(filename).split('_')[1]

    with open(filename, 'r') as file:
        spectrum_data = file.readlines()

    meas_date = spectrum_data[9].replace('START_TIME - ', '').strip()
    
    return meas_date, source



def extract_eff_info(filename):
    """
    Function that reads any supported spectrum file and returns start time
    and source type throughthe functions it calls.
    """
    cutis_folder = Path(__file__).parent / "CUITS"
   
    # Create the complete path to the specific file
    full_path = cutis_folder / filename
    # Extract the file extension (e.g., '.mca')
    suffix = full_path.suffix.lower()
    #suffix = Path(filename).suffix.lower()
    if suffix == ".spe":
        return eff_info_spe(full_path)
    if suffix == ".mca":
        return eff_info_mca(full_path)
    
def source_age(meas_date):
    '''
    Function that calculates the age of the source from the time that the
    source was calibrated and the time it was measured.
    '''
    
    # all measurements taken from upstairs sources.  
    calibration_date = '1979/02/01 12:00:00'

    cal_time = datetime.strptime(calibration_date, "%Y/%m/%d %H:%M:%S")
    meas_time = datetime.strptime(meas_date, "%m/%d/%Y %H:%M:%S")

    age_of_source = (meas_time - cal_time).total_seconds()
    
    return age_of_source


In [ ]:
def abs_efficiency(photo_peak, source_activity, time):
    '''

    assumes that the photo peak is in counts per second
    '''
    
    absolute_efficiency = photo_peak/(time*source_activity)

    return absolute_efficiency

In [ ]:
def intrinsic_eff(asb_effic, dist, radius):
    '''
    Function that calculates the intrinsic efficiency of the detector.
    This is done by determining and applying a geometry factor which 
    itself is dependant on the detectors radius and distance from 
    the radio isotope source.
    __________________
    Parameters
        dist: Float
        The distance the detector is from the radioisotope

        radius: Float
        radius of the detector

        abs_effic: Float
        absolute efficiency of the detector
    ___________________
    Outputs
        intrinsic_efficiency: Float
        intrinsic efficiency of the detector
    
    '''
    
    G = 0.5 * (1 - (dist / (sqrt((dist**2)+(radius**2))))) 
    
    intrinsic_efficiency = abs_effic / G
    
    return intrinsic_efficiency